In [ ]:
%pip install -r ../requirements.txt

In [ ]:
import os, json
import numpy as np
import scipy.sparse as sp
from pyscf import gto, scf
import sys; sys.path[:0] = ['..', '../data']  # repo root (srs/) and data/ (mol_info)
from srs.SRS_ray import SymmetryBasis, HamiltonianLoader, build_hamiltonian
from srs.encoding import stream_encode          # streaming Pauli encoder (srs/encoding.py)


def interleaved_to_blocked(x, M):
    """Mode map tau(2p) = p (alpha), tau(2p+1) = M + p (beta), applied bitwise."""
    x = np.asarray(x, dtype=np.int64)
    y = np.zeros_like(x)
    for p in range(M):
        y |= ((x >> (2 * p)) & 1) << p
        y |= ((x >> (2 * p + 1)) & 1) << (M + p)
    return y


def get_ham(Molecule, basis_sets='sto-3g', conv_tol=1e-12):
    """RHF reference (ROHF when spin != 0; Construction A) and integrals,
    relabelled to the blocked order alpha_0..alpha_{M-1} beta_0..beta_{M-1}."""
    mol = gto.M(atom=Molecule['atom'], basis=basis_sets,
                spin=Molecule['spin'], charge=Molecule['charge'], verbose=0)
    mf = scf.RHF(mol).run(conv_tol=conv_tol)
    if not mf.converged:
        raise RuntimeError("SCF not converged")
    M = mol.nao
    h1, h2, e_nuc = HamiltonianLoader(n_qubits=M).from_pyscf(mol, mf)
    tau = lambda k: k // 2 + M * (k % 2)
    h1B = {tuple(map(tau, k)): v for k, v in h1.items()}
    h2B = {tuple(map(tau, k)): v for k, v in h2.items()}
    return h1B, h2B, mol, mf, e_nuc


def build_block(h1B, h2B, mol, omega, mode='MHS'):
    """Restricted Hamiltonian (electronic) in the old Basis/Ham_setup order:
    ascending blocked integer label (colex)."""
    M, N = mol.nao, mol.nelectron
    states = np.asarray(SymmetryBasis(2 * M, N).build(mode=mode, multiplicity=omega))
    if states.size == 0:
        raise ValueError(f"{mode}(omega={omega}) is empty for N={N}, M={M}")
    basisB = np.sort(interleaved_to_blocked(states, M))
    Hb = build_hamiltonian(h1B, h2B, basisB, show=False)
    H = sp.csr_matrix((np.asarray(Hb.data), np.asarray(Hb.indices),
                       np.asarray(Hb.indptr)), shape=Hb.shape)
    if H.dtype != np.float64:
        raise ValueError(f"H has dtype {H.dtype}")
    return H, basisB


def auto_MHS_sections(Molecule, name='test', omegas=(0, 2), write=True,
                      shifts=('zero', 'mindiag'), outdir='encoded_hamiltonians',
                      export=True, n_workers=1, conv_tol=1e-12):
    """SRS_ray + stream_encode replacement of the qiskit-based version.
    export=False: numbers only (required for large D such as H22)."""
    h1B, h2B, mol, mf, e_nuc = get_ham(Molecule, conv_tol=conv_tol)
    records = []
    if write:
        os.makedirs(outdir, exist_ok=True)
    for omega in omegas:
        try:
            H, _ = build_block(h1B, h2B, mol, omega)
        except ValueError as e:
            print(f"  omega={omega}: skipped -- {e}")
            continue
        w = np.linalg.eigvalsh(H.toarray()) if H.shape[0] <= 4096 else None
        for shift in shifts:
            tag = '' if shift == 'zero' else '_shiftMinDiag'
            fname = (os.path.join(outdir, f"{name}_MHS_omega{omega}{tag}.txt")
                     if (write and export) else None)
            out = stream_encode(H, pad=shift, export=fname,
                                n_workers=1 if fname else n_workers, report=False)
            assert out['L_noI'] >= out['floor'] - 1e-8
            rec = dict(molecule=name, omega=omega, S=omega / 2, shift=shift,
                       constant=e_nuc, nuc=e_nuc, eigenval=w, **out)
            records.append(rec)
            print(f"  omega={omega} shift={shift:7s} lam={out['lam']:+.6f}: D={out['D']} "
                  f"q={out['q']} T={out['T']} c_I={out['c_I']:+.8f} L_noI={out['L_noI']:.6f} "
                  f"floor={out['floor']:.6f} E0={out['E0']:.8f} N_XBM={out['N_XBM']} Z={out['Z']:.6f}")
    if write:
        rpath = os.path.join(outdir, f"{name}_records.json")
        with open(rpath, 'w') as f:
            json.dump(records, f, indent=2,
                      default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))
        print(f"saved {len(records)} records to '{rpath}'")
    return records

In [3]:
import mol_info as mi            # geometries: H2, LiH, ... are dicts in data/mol_info.py
from math import comb

MOLS = ["H2","LiH", 'HF', 'BeH2', 'H2O', 'OH', 'CH']
all_records = []
for name in MOLS:
    if not hasattr(mi, name):
        print(f"== {name}: not in molecule_info.py, skipped"); continue
    info = getattr(mi, name)
    mol = gto.M(atom=info["atom"], basis="sto-3g", spin=info["spin"],
                charge=info["charge"], verbose=0)
    M, N = mol.nao, mol.nelectron
    omegas = []
    for om in range(N % 2, min(N, 2*M - N) + 1, 2):
        na, nb = (N + om)//2, (N - om)//2
        D = comb(M, na) * comb(na, nb)          # dim MHS(omega)
        omegas.append(om)
    print(name)
    auto_MHS_sections(info, name =name, omegas=tuple(omegas), outdir='hamiltonians')
    print()

H2
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=2 q=1 T=3 c_I=-1.04247102 L_noI=0.969256 floor=0.808553 E0=-1.85102417 N_XBM=2 Z=2.011727
  omega=0 shift=mindiag lam=-1.830438: D=2 q=1 T=3 c_I=-1.04247102 L_noI=0.969256 floor=0.808553 E0=-1.85102417 N_XBM=2 Z=2.011727
use tree
  omega=2 shift=zero    lam=+0.000000: D=1 q=0 T=1 c_I=-1.24623300 L_noI=0.000000 floor=0.000000 E0=-1.24623300 N_XBM=1 Z=1.246233
  omega=2 shift=mindiag lam=+0.000000: D=1 q=0 T=1 c_I=-1.24623300 L_noI=0.000000 floor=0.000000 E0=-1.24623300 N_XBM=1 Z=1.246233
saved 4 records to 'hamiltonians/H2_records.json'

LiH
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=15 q=4 T=134 c_I=-4.35575316 L_noI=9.545443 floor=3.291630 E0=-8.87338861 N_XBM=16 Z=9.615888
  omega=0 shift=mindiag lam=-8.857407: D=15 q=4 T=134 c_I=-4.90934110 L_noI=9.094932 floor=3.291630 E0=-8.87338861 N_XBM=16 Z=9.615888
use tree
  omega=2 shift=zero    lam=+0.000000: D=60 

In [ ]:
'''
H2
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=2 q=1 T=3 c_I=-1.04247102 L_noI=0.969256 floor=0.808553 E0=-1.85102417 N_XBM=2 Z=2.011727
  omega=0 shift=mindiag lam=-1.830438: D=2 q=1 T=3 c_I=-1.04247102 L_noI=0.969256 floor=0.808553 E0=-1.85102417 N_XBM=2 Z=2.011727
use tree
  omega=2 shift=zero    lam=+0.000000: D=1 q=0 T=1 c_I=-1.24623300 L_noI=0.000000 floor=0.000000 E0=-1.24623300 N_XBM=1 Z=1.246233
  omega=2 shift=mindiag lam=+0.000000: D=1 q=0 T=1 c_I=-1.24623300 L_noI=0.000000 floor=0.000000 E0=-1.24623300 N_XBM=1 Z=1.246233
saved 4 records to 'hamiltonians/H2_records.json'

LiH
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=15 q=4 T=134 c_I=-4.35575316 L_noI=9.545443 floor=3.291630 E0=-8.87338861 N_XBM=16 Z=9.615888
  omega=0 shift=mindiag lam=-8.857407: D=15 q=4 T=134 c_I=-4.90934110 L_noI=9.094932 floor=3.291630 E0=-8.87338861 N_XBM=16 Z=9.615888
use tree
  omega=2 shift=zero    lam=+0.000000: D=60 q=6 T=2080 c_I=-4.64107725 L_noI=18.277274 floor=3.177226 E0=-8.76150437 N_XBM=64 Z=12.747319
  omega=2 shift=mindiag lam=-8.717196: D=60 q=6 T=2080 c_I=-5.18590202 L_noI=16.665828 floor=3.177226 E0=-8.76150437 N_XBM=64 Z=12.747319
use tree
  omega=4 shift=zero    lam=+0.000000: D=15 q=4 T=120 c_I=-4.97793300 L_noI=7.469137 floor=1.909689 E0=-6.69887234 N_XBM=14 Z=7.613705
  omega=4 shift=mindiag lam=-6.672360: D=15 q=4 T=120 c_I=-5.39495547 L_noI=5.510004 floor=1.909689 E0=-6.69887234 N_XBM=14 Z=7.613705
saved 6 records to 'hamiltonians/LiH_records.json'

HF
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=6 q=3 T=36 c_I=-69.93169196 L_noI=98.638142 floor=27.823981 E0=-103.78651942 N_XBM=8 Z=104.658590
  omega=0 shift=mindiag lam=-103.765560: D=6 q=3 T=36 c_I=-95.87308197 L_noI=48.627392 floor=27.823981 E0=-103.78651942 N_XBM=8 Z=104.658590
use tree
  omega=2 shift=zero    lam=+0.000000: D=15 q=4 T=120 c_I=-88.17940299 L_noI=92.337842 floor=13.540167 E0=-103.35696034 N_XBM=14 Z=104.722307
  omega=2 shift=mindiag lam=-103.325323: D=15 q=4 T=120 c_I=-94.63723567 L_noI=37.500128 floor=13.540167 E0=-103.35696034 N_XBM=14 Z=104.722307
saved 4 records to 'hamiltonians/HF_records.json'

BeH2
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=35 q=6 T=1612 c_I=-6.42922213 L_noI=39.043986 floor=6.393267 E0=-18.96985571 N_XBM=51 Z=22.657940
  omega=0 shift=mindiag lam=-18.951451: D=35 q=6 T=1612 c_I=-15.01659836 L_noI=31.355710 floor=6.393267 E0=-18.96985571 N_XBM=51 Z=22.657940
use tree
  omega=2 shift=zero    lam=+0.000000: D=210 q=8 T=32192 c_I=-9.90480346 L_noI=84.570688 floor=6.167161 E0=-18.68821234 N_XBM=252 Z=41.759881
  omega=2 shift=mindiag lam=-18.674312: D=210 q=8 T=32192 c_I=-13.26034383 L_noI=75.185604 floor=6.167161 E0=-18.68821234 N_XBM=252 Z=41.759881
use tree
  omega=4 shift=zero    lam=+0.000000: D=105 q=7 T=7840 c_I=-10.24407145 L_noI=53.790880 floor=5.850567 E0=-18.50151202 N_XBM=122 Z=29.306930
  omega=4 shift=mindiag lam=-18.475299: D=105 q=7 T=7840 c_I=-13.56385168 L_noI=41.621734 floor=5.850567 E0=-18.50151202 N_XBM=122 Z=29.306930
use tree
  omega=6 shift=zero    lam=+0.000000: D=7 q=3 T=20 c_I=-11.37241297 L_noI=11.686627 floor=3.086440 E0=-14.36683136 N_XBM=4 Z=14.678602
  omega=6 shift=mindiag lam=-14.364387: D=7 q=3 T=20 c_I=-13.16796140 L_noI=5.281589 floor=3.086440 E0=-14.36683136 N_XBM=4 Z=14.678602
saved 8 records to 'hamiltonians/BeH2_records.json'

H2O
[Loader] Loading from PySCF (RHF)
use tree
  omega=0 shift=zero    lam=+0.000000: D=21 q=5 T=504 c_I=-45.57356382 L_noI=154.122433 floor=23.603451 E0=-84.17763227 N_XBM=31 Z=87.287646
  omega=0 shift=mindiag lam=-84.152557: D=21 q=5 T=504 c_I=-74.50100525 L_noI=74.239484 floor=23.603451 E0=-84.17763227 N_XBM=31 Z=87.287646
use tree
  omega=2 shift=zero    lam=+0.000000: D=105 q=7 T=8096 c_I=-57.49059952 L_noI=227.139311 floor=23.101139 E0=-83.77091322 N_XBM=126 Z=102.075463
  omega=2 shift=mindiag lam=-83.745097: D=105 q=7 T=8096 c_I=-72.53854655 L_noI=149.090866 floor=23.101139 E0=-83.77091322 N_XBM=126 Z=102.075463
use tree
  omega=4 shift=zero    lam=+0.000000: D=35 q=6 T=1456 c_I=-38.75031800 L_noI=147.267643 floor=11.138685 E0=-83.25549117 N_XBM=45 Z=88.719769
  omega=4 shift=mindiag lam=-83.208777: D=35 q=6 T=1456 c_I=-76.45429503 L_noI=58.125738 floor=11.138685 E0=-83.25549117 N_XBM=45 Z=88.719769
saved 6 records to 'hamiltonians/H2O_records.json'

OH
[Loader] Loading from PySCF (RHF)
use tree
  omega=1 shift=zero    lam=+0.000000: D=30 q=5 T=504 c_I=-62.77424719 L_noI=109.103835 floor=22.385127 E0=-78.74492479 N_XBM=31 Z=82.650304
  omega=1 shift=mindiag lam=-78.727229: D=30 q=5 T=504 c_I=-67.69469900 L_noI=74.917268 floor=22.385127 E0=-78.74492479 N_XBM=31 Z=82.650304
use tree
  omega=3 shift=zero    lam=+0.000000: D=20 q=5 T=272 c_I=-42.34724021 L_noI=102.967683 floor=10.620379 E0=-78.35908641 N_XBM=17 Z=80.129207
  omega=3 shift=mindiag lam=-78.333701: D=20 q=5 T=272 c_I=-71.72237801 L_noI=38.645205 floor=10.620379 E0=-78.35908641 N_XBM=17 Z=80.129207
saved 4 records to 'hamiltonians/OH_records.json'

CH
[Loader] Loading from PySCF (RHF)
use tree
  omega=1 shift=zero    lam=+0.000000: D=60 q=6 T=1872 c_I=-28.05705888 L_noI=86.273236 floor=12.715259 E0=-40.62895820 N_XBM=59 Z=46.364226
  omega=1 shift=mindiag lam=-40.603856: D=60 q=6 T=1872 c_I=-30.59479989 L_noI=68.257157 floor=12.715259 E0=-40.62895820 N_XBM=59 Z=46.364226
use tree
  omega=3 shift=zero    lam=+0.000000: D=60 q=6 T=2072 c_I=-28.62458403 L_noI=65.891445 floor=12.434031 E0=-40.63675350 N_XBM=64 Z=46.899747
  omega=3 shift=mindiag lam=-40.605669: D=60 q=6 T=2072 c_I=-31.16243834 L_noI=49.977301 floor=12.434031 E0=-40.63675350 N_XBM=64 Z=46.899747
use tree
  omega=5 shift=zero    lam=+0.000000: D=6 q=3 T=32 c_I=-23.43874596 L_noI=29.379460 floor=5.556553 E0=-40.00331298 N_XBM=7 Z=40.416639
  omega=5 shift=mindiag lam=-40.003265: D=6 q=3 T=32 c_I=-33.43956227 L_noI=12.504417 floor=5.556553 E0=-40.00331298 N_XBM=7 Z=40.416639
saved 6 records to 'hamiltonians/CH_records.json'

'''